# 1. Core Pipeline — Fetch & Quantify

This notebook __produces the data__ used by the other two. It:
1. Fetches OHLC data from Bybit for a given symbol, interval, and date range.
2. Computes per-EMA metrics: touches, crosses, above/below counts, evaluated candles.
3. Caches OHLC to data/ as parquet (keyed by config). Notebooks 2 & 3 reuse it instead of re-fetching.
4. Writes data/config.json so notebooks 2 & 3 pick up the same parameters automatically — change them here, not there.

## Table of contents

1. [Project Overview](#Project-Overview)
2. [Cache and re-fetching](#Cache-and-re-fetching)
3. [Setup](#Setup)
4. [Configuration](#Configuration)
5. [Fetch OHLC](#Fetch-OHLC)
6. [Aggregate EMA metrics](#Aggregate-EMA-metrics)
7. [Sanity check](#Sanity-check)
8. [Data export](#Data-export)
9. [Next steps](#Next-steps)

## Project Overview

__Data:__ \
ByBit Perpetual Futures.

__Workflow:__ \
For each EMA period in a user-supplied range, every evaluated candle is placed in exactly one of five mutually-exclusive categories, so the categories always sum to evaluated_candles.

__Inputs:__
- symbol
- interval (ByBit codes: 1/3/5/15/30/60/120/240/360/720/D/W/M)
- start/end dates (YYYY-MM-DD)
- ema_range (any iterable of integers, e.g. range(1, 101) or [9, 21, 50, 200])
- delta
- delta_mode

__Delta modes:__
- "percent" — delta is a % of the EMA value at each candle (recommended, scales across price regimes) \
The allowed distance changes with price.
- "absolute" — delta is a fixed price distance in quote currency (e.g. USDT). Distance in price units. \
The allowed distance is always exactly the chosen 'number' regardless of price.

__Categories__ (mutually exclusive partition; every candle lands in one category):
- cross      → Low ≤ EMA ≤ High. The candle's range straddles the EMA — the level was *broken*, not held.
- low_touch  → Low > EMA AND Low − EMA ≤ delta. Entire candle above EMA; low approached EMA from above without crossing — a *test of support*.
- high_touch → High < EMA AND EMA − High ≤ delta. Entire candle below EMA; high approached EMA from below without crossing — a *test of resistance*.
- above      → Low > EMA AND Low − EMA > delta. Entire candle above EMA, low far away — clean uptrend, no test.
- below      → High < EMA AND EMA − High > delta. Entire candle below EMA, high far away — clean downtrend, no test.

__Invariants:__
- cross + low_touch + high_touch + above + below = evaluated_candles
- any_touch = low_touch + high_touch (the two are mutually exclusive — a candle either tests support OR resistance, never both)

__Notes:__
- The first period candles are skipped per EMA (warm-up); toggle with skip_warmup=False if you don't want that.
- Pagination handles arbitrarily long date ranges (1000-candle chunks).
- Uses ewm(span=period, adjust=False) — standard Wilder/TradingView-style EMA.


## Cache and re-fetching

__What gets cached:__ \
Raw OHLC from ByBit: one parquet file under data/klines_{symbol}_{interval}_{start}_{end}_{category}.parquet.

__Default behavior__ (load_or_fetch_klines):
- If a klines file exists for your config → load from cache
- Otherwise → fetch from Bybit and save
- First run (new market): seconds to minutes
- Subsequent runs: instant

__What happens when you change parameters:__
| change | klines cache | what runs |
|---|---|---|
| symbol, interval, start, end, category | new entry | full pipeline (ByBit fetch + analyze) |
| ema_range, delta, delta_mode, skip_warmup | unchanged | analyze only (no fetch) |


__Configuration:__
- The Configuration block at the top of each notebook defines what is fetched and analyzed.
- It also determines the cache key.
- Use the same Configuration in all three notebooks — otherwise notebooks 2 & 3 will miss the cache and re-fetch data.
- __Important:__ when you change a parameter, update it in all three notebooks before running.

__Cache management:__
- Old klines files are not deleted automatically
- Clean them up with: rm data/*.parquet

__Force a refresh:__
- Use this if data may have changed (ByBit corrected historical data) or to test the cache logic itself:
```python
df = load_or_fetch_klines(symbol, interval, start, end, category, force_refetch=True)
```
- This re-fetches from Bybit and overwrites the existing cache file.


Notebooks 2 and 3 read from the same klines cache notebook 1 wrote.  **If the configuration differs across notebooks, the cache lookup fails** and notebook 2 / 3 will silently re-fetch their own copy.



## Setup


In [ ]:
from ema_core import load_or_fetch_klines, analyze_ema_touches, save_config


## Configuration

Edit these to fetch a different market / range. Anything you change here also needs to match in notebooks 2 & 3 — they use the same values to find the cache files.

Configuration:
- interval = "60" for 1-hour candles
- ema_range  = range(5, 201, 5) calculates EMA 5, 10, 15, ..., 200
- delta_mode = "percent" or "absolute"
- delta %: the allowed distance between EMA and candle changes with price \
  delta=0.5 → "low within 0.5% of EMA"
- delta absolute: the allowed distance is always exactly the chosen 'number' regardless of price. Distance in price units. \
  delta=40 → "low within $40 of EMA"

*Examples:*
- delta=0.10 is 0.10 % of the EMA 
- a 5 USDT price difference in % is calculated as:  5 / price (e.g. 67,000) × 100 = 0.00746%
- a 0.0075% delta in USDT is calculated as: price (e.g. 67,000) × 0.0075 / 100 = 5.025 USDT

In [ ]:
symbol     = "BTCUSDT"
interval   = "15"
start      = "2026-01-01"
end        = "2026-04-01"
ema_range  = range(1, 200, 1)
delta      = 20
delta_mode = "absolute"          # "percent" (of EMA) or "absolute" (quote ccy)
category   = "linear"            # "linear" (USDT perps) or "inverse" (coin-margined)


# Persist this config to data/config.json so notebooks 2 & 3 read it via load_config().
# Edit values above and re-run this cell to update.
save_config(symbol, interval, start, end, ema_range, delta, delta_mode, category)


## Fetch OHLC

load_or_fetch_klines returns the cached parquet if present, otherwise fetches from ByBit and caches the result. \
Pass force_refetch=True to refresh.

Dataframe of raw OHLC candles (klines) from ByBit:

In [ ]:
df = load_or_fetch_klines(symbol, interval, start, end, category)
print(f"  -> {len(df)} candles loaded, interval={interval}, {df['timestamp'].iloc[0]} → {df['timestamp'].iloc[-1]}")
df.head()


## Aggregate EMA metrics

analyze_ema_touches runs the per-EMA touch / cross / above / below counting on the OHLC data. \
For each period in ema_range it computes ewm(span=period, adjust=False) and tallies the partition + quality counters described in the docstring.

delta_mode  : "percent" (of EMA value) or "absolute" (fixed quote-currency) \
delta %: the allowed distance changes with price \
delta absolute: the allowed distance is always exactly the chosen 'number' regardless of price


Main algorithm:
1. Fetches ByBit perpetual futures data.
2. For every EMA period in ema_range:
   - Computes the EMA on close prices.
   - Assigns every evaluated candle to a category.
   - Categories are mutually exclusive.
3. Returns a clean DataFrame with one row per EMA period.


Partition table: every evaluated candle is placed in exactly one category.

| category     | condition                                                            | behavior | meaning |
|--------------|----------------------------------------------------------------------|---------|----------|
| cross      | Low ≤ EMA ≤ High | candle crosses EMA | level broken             |
| low_touch  | Low > EMA AND Low − EMA ≤ delta | entire candle above EMA, Low within delta of EMA, Low approached EMA from above | support held, no break |
| high_touch | High < EMA AND EMA − High ≤ delta | entire candle below EMA, High within delta of EMA, High approached EMA from below | resistance held, no break |
| above      | Low > EMA AND Low − EMA > delta | entire candle above EMA, Low further than delta | clean uptrend, no test of EMA |
| below      | High < EMA AND EMA − High > delta | entire candle below EMA, High further than delta | clean downtrend, no test of EMA |

Invariants:
- cross + low_touch + high_touch + above + below = evaluated_candles
- any_touch = low_touch + high_touch (mutually exclusive — a candle either tests support OR resistance, never both)


Dataframe of candle categories by EMA period:

In [ ]:
result = analyze_ema_touches(df, ema_range, delta, delta_mode)
print(f"  -> {len(result)} EMA periods analyzed.")
result.head(10).style.hide(axis="index")


__Alternative: convenience wrapper__

Convenience wrapper:
- chains fetch + analyze in one call
- does NOT use the parquet cache (always fetches from ByBit)
- for cached fetch, use load_or_fetch_klines + analyze_ema_touches instead (what this notebook does)
- returns 2 dataframes: df, result


In [ ]:
from ema_core import run
df, result = run(symbol, interval, start, end, ema_range, delta, delta_mode)

## Sanity check

A quick look at the partition counts for a handful of EMAs. \
The invariant cross + low_touch + high_touch + above + below = evaluated_candles should hold for every row.

In [ ]:
sample = result.iloc[::max(1, len(result) // 10)][['ema', 'low_touch', 'high_touch', 'cross', 'above', 'below', 'evaluated_candles']]
sample.assign(_sum=sample[['low_touch','high_touch','cross','above','below']].sum(axis=1)).head(10)


## Data export

Saving dataframes to CSV for further analysis in Excel or Tableau.

In [ ]:
filename_df = f"df_{symbol}_{interval}_{start}_{end}.csv"
df.to_csv(filename_df, index=False)
print(f"\nResults saved to {filename_df}")

In [ ]:
filename_ema = f"ema_analysis_{symbol}_{interval}_{start}_{end}_{ema_range}_{delta}.csv"
result.to_csv(filename_ema, index=False)
print(f"\nResults saved to {filename_ema}")

## Next steps

1. Both cached dataframes are now on disk under data/.
2. The other 2 notebooks automatically load the cached data if the configuration matches.
3. Open 2_ema_analysis.ipynb next to interpret which EMAs act as support / resistance / universal S/R.
4. Or skip ahead to 3_ema_backtesting.ipynb to backtest a strategy on this dataset.

__Practical workflow__

| if you want | run |
|---|---|
| to explore the data without trading anything | 1 → 2 |
| to understand the EMA landscape before committing to a strategy | 1 → 2 → 3 |
| just to check whether a strategy is profitable on this market right now | 1 → 3 (skip 2) |
